Cell 0 — Fix environment + install deps (run once, then RESTART kernel)

In [1]:
import sys, subprocess

def pip(cmd):
    print(">", "pip", cmd)
    subprocess.check_call([sys.executable, "-m", "pip"] + cmd.split())

pip("install -U -q pip setuptools wheel")

# Repair numpy/scipy/sklearn if your env is broken (you had _center import errors before)
def sci_ok():
    try:
        import numpy as np
        import scipy
        import sklearn
        return True
    except Exception as e:
        print("⚠️ scientific stack broken:", repr(e))
        return False

if not sci_ok():
    pip("uninstall -y -q numpy scipy scikit-learn")
    pip("install -q --no-cache-dir --force-reinstall numpy==1.26.4 scipy==1.12.0 scikit-learn==1.4.2 pandas==2.2.2")

# HF stack
pip("install -q -U transformers==4.45.2 datasets==2.21.0 accelerate==0.34.2 evaluate==0.4.2 sentencepiece")

# For 4-bit LLM (Kaggle GPU only)
try:
    import bitsandbytes  # noqa
except Exception:
    try:
        pip("install -q -U bitsandbytes")
    except Exception:
        print("⚠️ bitsandbytes not available (CPU-only). We'll fall back to a smaller explainer.")

print("✅ Done. Now RESTART the runtime/kernel, then run the next cell.")


> pip install -U -q pip setuptools wheel
> pip install -q -U transformers==4.45.2 datasets==2.21.0 accelerate==0.34.2 evaluate==0.4.2 sentencepiece
✅ Done. Now RESTART the runtime/kernel, then run the next cell.


Cell 1 — Config

In [2]:
from pathlib import Path

# --- Paths ---
DATA_ROOT = "/kaggle/input"
LOCAL_TRAIN_FALLBACK = "/mnt/data/train.csv"   # if you run locally

# --- High-accuracy classifier ---
CLASSIFIER_MODEL = "microsoft/deberta-v3-large"

# --- Best explainer LLM (teacher feedback) ---
EXPLAINER_MODEL = "Qwen/Qwen2.5-7B-Instruct"        # open + strong
EXPLAINER_FALLBACK_MODEL = "google/flan-t5-large"   # if no 4-bit GPU

# --- Training ---
N_FOLDS = 5
SEEDS = [42]               # you can do [42, 123] for better CV (slower)
EPOCHS = 2                 # try 3 for stronger
LR = 2e-5
BATCH_SIZE = 4             # DeBERTa-large is heavy. Use 2-4 on T4.
GRAD_ACCUM = 4
MAX_LEN = 384              # try 512 if memory allows
WEIGHT_DECAY = 0.01
USE_FP16 = True

# --- Accuracy boosts (safe vs risky) ---
USE_FAMILY_FILTER = True            # uses inferred correctness to filter True/False label families (usually helps)
USE_QID_LABEL_RESTRICTION = False   # can inflate CV; keep False unless you explicitly want it

# --- Explainability ---
USE_LLM_EXPLAINER = True
USE_LLM_VERIFIER = True
KB_BUILD_TOP_N = 200
KB_EXAMPLES_PER_MISCON = 6

# --- Cache ---
WORKDIR = Path("./")
KB_PATH = WORKDIR / "misconception_kb.json"


Cell 2 — Load train.csv + detect columns

In [3]:
import pandas as pd

def find_file(filename: str, root: str) -> str:
    rootp = Path(root)
    hits = sorted(rootp.rglob(filename), key=lambda p: len(str(p)))
    return str(hits[0]) if hits else ""

train_path = find_file("train.csv", DATA_ROOT)
if not train_path and Path(LOCAL_TRAIN_FALLBACK).exists():
    train_path = LOCAL_TRAIN_FALLBACK
if not train_path:
    raise FileNotFoundError("train.csv not found. Put it in /kaggle/input/** or set LOCAL_TRAIN_FALLBACK.")

train_df = pd.read_csv(train_path)
print("train:", train_path, train_df.shape)
print("cols:", list(train_df.columns))

def lower_map(cols): return {c.lower(): c for c in cols}
cols = lower_map(train_df.columns)

cat_col = cols.get("category")
mis_col = cols.get("misconception")
if cat_col is None or mis_col is None:
    raise ValueError("Need columns: Category, Misconception")

qid_col = None
for k in ["question_id","questionid","item_id","itemid","problem_id","problemid","stem_id","stemid"]:
    if k in cols: qid_col = cols[k]; break

ans_col = None
for k in ["mc_answer","answer_choice","answer","choice","selected_answer","student_answer","option"]:
    if k in cols: ans_col = cols[k]; break

exp_col = None
for k in ["student_explanation","explanation","student_response","response","open_response","rationale"]:
    if k in cols: exp_col = cols[k]; break

qtext_col = None
for k in ["question_text","question","prompt","stem","item_stem","problem","task"]:
    if k in cols: qtext_col = cols[k]; break

print("Detected:")
print(" category:", cat_col)
print(" misconception:", mis_col)
print(" qid:", qid_col)
print(" answer:", ans_col)
print(" explanation:", exp_col)
print(" question text:", qtext_col)

train_df["label_str"] = train_df[cat_col].astype(str) + ":" + train_df[mis_col].astype(str)
train_df.head(3)


train: /kaggle/input/competitions/map-charting-student-math-misunderstandings/train.csv (36696, 7)
cols: ['row_id', 'QuestionId', 'QuestionText', 'MC_Answer', 'StudentExplanation', 'Category', 'Misconception']
Detected:
 category: Category
 misconception: Misconception
 qid: QuestionId
 answer: MC_Answer
 explanation: None
 question text: None


,row_id,QuestionId,QuestionText,MC_Answer,StudentExplanation,Category,Misconception,label_str
0,0,31772,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),0ne third is equal to tree nineth,True_Correct,NaN,True_Correct:nan
1,1,31772,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),1 / 3 because 6 over 9 is 2 thirds and 1 third...,True_Correct,NaN,True_Correct:nan
2,2,31772,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),"1 3rd is half of 3 6th, so it is simplee to un...",True_Neither,NaN,True_Neither:nan


Cell 3 — Infer correct answer per question + build model input text

In [4]:
import numpy as np

def build_correct_by_q(df: pd.DataFrame) -> dict:
    if qid_col is None or ans_col is None:
        return {}
    tcat = df[cat_col].astype(str)
    correct_mask = tcat.str.startswith("True", na=False) | (tcat == "True_Correct")
    tmp = df.loc[correct_mask, [qid_col, ans_col]].dropna()
    if len(tmp) == 0:
        return {}
    return tmp.groupby(qid_col)[ans_col].agg(lambda s: s.value_counts().index[0]).to_dict()

correct_by_q_full = build_correct_by_q(train_df)
print("Correct answers inferred:", len(correct_by_q_full))

def compute_is_correct(df: pd.DataFrame, correct_by_q: dict) -> np.ndarray:
    if qid_col is None or ans_col is None or not correct_by_q:
        return np.full(len(df), -1, dtype=np.int8)
    out = np.full(len(df), -1, dtype=np.int8)
    for i, (q, a) in enumerate(zip(df[qid_col].values, df[ans_col].values)):
        if pd.isna(q) or pd.isna(a): 
            continue
        if q in correct_by_q:
            out[i] = 1 if str(a) == str(correct_by_q[q]) else 0
    return out

train_df["is_correct"] = compute_is_correct(train_df, correct_by_q_full)
print(train_df["is_correct"].value_counts(dropna=False))

def safe_col(df, col):
    if col is None or col not in df.columns:
        return pd.Series([""] * len(df))
    return df[col].fillna("").astype(str)

def make_text(df: pd.DataFrame, is_correct: np.ndarray) -> list:
    q = safe_col(df, qtext_col)
    a = safe_col(df, ans_col)
    e = safe_col(df, exp_col)
    ic = pd.Series(is_correct).map({1:"Yes", 0:"No"}).fillna("Unknown").astype(str)
    return ("Question: " + q + "\nAnswer: " + a + "\nIs Correct Answer: " + ic + "\nStudent Explanation: " + e).tolist()

train_text_full = make_text(train_df, train_df["is_correct"].values.astype(np.int8))
print(train_text_full[0][:500])


Correct answers inferred: 15
is_correct
1    20464
0    16232
Name: count, dtype: int64
Question: 
Answer: \( \frac{1}{3} \)
Is Correct Answer: Yes
Student Explanation: 


Cell 4 — Train DeBERTa + CV MAP@3 (mean score on train)

In [ ]:
import torch
from sklearn.model_selection import GroupKFold
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding, set_seed
)
from datasets import Dataset

classes = np.array(sorted(train_df["label_str"].unique().tolist()))
lab2id = {c:i for i,c in enumerate(classes)}
id2lab = {i:c for c,i in lab2id.items()}
y_all = train_df["label_str"].map(lab2id).astype(int).values
num_labels = len(classes)
print("num_labels:", num_labels)

true_mask  = np.array([c.startswith("True")  for c in classes], dtype=bool)
false_mask = np.array([c.startswith("False") for c in classes], dtype=bool)

def apply_filters(logits: np.ndarray, is_correct: np.ndarray, allowed_mask=None) -> np.ndarray:
    out = logits.copy()
    big_neg = -1e9
    if USE_FAMILY_FILTER:
        for i, ic in enumerate(is_correct):
            if ic == 1: out[i, ~true_mask] = big_neg
            elif ic == 0: out[i, ~false_mask] = big_neg
    if allowed_mask is not None:
        out[~allowed_mask] = big_neg
    return out

def top3_from_logits(logits: np.ndarray):
    idx = np.argsort(-logits, axis=1)[:, :10]
    out = []
    for row in idx:
        seen, picks = set(), []
        for j in row:
            if j in seen: 
                continue
            seen.add(int(j))
            picks.append(int(j))
            if len(picks) == 3:
                break
        while len(picks) < 3:
            picks.append(int(row[len(picks)]))
        out.append(picks)
    return out

def map_at_3(y_true: np.ndarray, y_pred_top3):
    score = 0.0
    for yt, preds in zip(y_true, y_pred_top3):
        used = set()
        for rank, p in enumerate(preds[:3], start=1):
            if p in used: 
                continue
            used.add(p)
            if p == yt:
                score += 1.0 / rank
                break
    return score / len(y_true)

tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL, use_fast=True)
collator = DataCollatorWithPadding(tokenizer)

groups = train_df[qid_col].values if qid_col is not None else np.arange(len(train_df))
gkf = GroupKFold(n_splits=N_FOLDS)

fold_scores = []
oof_top3_labels = [None] * len(train_df)

for seed in SEEDS:
    set_seed(seed)
    print(f"\n==================== SEED {seed} ====================")

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(np.zeros(len(train_df)), y_all, groups), start=1):
        tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
        va_df = train_df.iloc[va_idx].reset_index(drop=True)

        # avoid leakage: infer correct answers from TRAIN fold only
        correct_by_q = build_correct_by_q(tr_df)
        tr_is_correct = compute_is_correct(tr_df, correct_by_q)
        va_is_correct = compute_is_correct(va_df, correct_by_q)

        allowed_mask_va = None
        if USE_QID_LABEL_RESTRICTION and qid_col is not None:
            allowed_by_q = {}
            tr_y = tr_df["label_str"].map(lab2id).values
            for q, lab in zip(tr_df[qid_col].values, tr_y):
                allowed_by_q.setdefault(q, set()).add(int(lab))
            allowed_mask_va = np.zeros((len(va_df), num_labels), dtype=bool)
            for i, q in enumerate(va_df[qid_col].values):
                if q in allowed_by_q:
                    allowed_mask_va[i, list(allowed_by_q[q])] = True
                else:
                    allowed_mask_va[i, :] = True

        tr_text = make_text(tr_df, tr_is_correct)
        va_text = make_text(va_df, va_is_correct)

        tr_y = tr_df["label_str"].map(lab2id).astype(int).values
        va_y = va_df["label_str"].map(lab2id).astype(int).values

        tr_ds = Dataset.from_dict({"text": tr_text, "label": tr_y, "is_correct": tr_is_correct.tolist()})
        va_ds = Dataset.from_dict({"text": va_text, "label": va_y, "is_correct": va_is_correct.tolist()})

        def tok(batch): return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)
        tr_ds = tr_ds.map(tok, batched=True, remove_columns=["text"]).with_format("torch")
        va_ds = va_ds.map(tok, batched=True, remove_columns=["text"]).with_format("torch")

        model = AutoModelForSequenceClassification.from_pretrained(CLASSIFIER_MODEL, num_labels=num_labels)

        args = TrainingArguments(
            output_dir=f"./tmp_cls_seed{seed}_fold{fold}",
            learning_rate=LR,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE*2,
            gradient_accumulation_steps=GRAD_ACCUM,
            num_train_epochs=EPOCHS,
            weight_decay=WEIGHT_DECAY,
            fp16=(USE_FP16 and torch.cuda.is_available()),
            logging_steps=100,
            save_strategy="no",
            evaluation_strategy="no",
            report_to="none",
            seed=seed,
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=tr_ds,
            eval_dataset=va_ds,
            tokenizer=tokenizer,
            data_collator=collator,
        )

        print(f"\n--- Fold {fold}/{N_FOLDS} ---")
        trainer.train()

        pred = trainer.predict(va_ds)
        va_logits = np.asarray(pred.predictions, dtype=np.float32)
        va_logits = apply_filters(va_logits, va_is_correct, allowed_mask_va)

        va_top3 = top3_from_logits(va_logits)
        fold_map3 = map_at_3(va_y, va_top3)
        fold_scores.append(fold_map3)
        print("Fold MAP@3:", round(fold_map3, 6))

        for ii, ridx in enumerate(va_idx):
            oof_top3_labels[ridx] = [id2lab[j] for j in va_top3[ii]]

print("\n==================== TRAIN RESULTS ====================")
print("Fold scores:", [round(s, 6) for s in fold_scores])
print("Mean MAP@3 :", round(float(np.mean(fold_scores)), 6))

train_df["pred_top3"] = [" ".join(p) for p in oof_top3_labels]
train_df[["label_str","pred_top3"]].head(5)


2026-02-19 10:25:04.377538: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771496704.574025     195 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771496704.626981     195 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771496705.060944     195 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771496705.060982     195 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771496705.060985     195 computation_placer.cc:177] computation placer alr

num_labels: 65


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:558: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


... (4 outputs left out)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)



--- Fold 1/5 ---


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
100,2.008800


Cell 5 — Load Explainer LLM (Qwen 7B 4-bit) + helper generate()

In [ ]:
import torch

def load_explainer():
    if not USE_LLM_EXPLAINER:
        return None, None, None

    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM,
        AutoModelForSeq2SeqLM,
        BitsAndBytesConfig
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Best: 4-bit causal LLM (GPU)
    if device == "cuda":
        try:
            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
            tok = AutoTokenizer.from_pretrained(EXPLAINER_MODEL, use_fast=True)
            mdl = AutoModelForCausalLM.from_pretrained(
                EXPLAINER_MODEL,
                quantization_config=bnb_cfg,
                device_map="auto",
                torch_dtype=torch.float16,
            )
            mdl.eval()
            print("✅ Loaded explainer (4-bit):", EXPLAINER_MODEL)
            return "causal", mdl, tok
        except Exception as e:
            print("⚠️ 4-bit causal explainer failed:", repr(e)[:200])

    # Fallback: FLAN-T5 (CPU/GPU)
    tok = AutoTokenizer.from_pretrained(EXPLAINER_FALLBACK_MODEL, use_fast=True)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(EXPLAINER_FALLBACK_MODEL).to(device)
    mdl.eval()
    print("✅ Loaded explainer fallback:", EXPLAINER_FALLBACK_MODEL, "on", device)
    return "seq2seq", mdl, tok

EXPLAINER_KIND, explainer_model, explainer_tok = load_explainer()

def llm_generate(prompt: str, max_new_tokens=320):
    if explainer_model is None:
        return ""
    if EXPLAINER_KIND == "causal":
        inputs = explainer_tok(prompt, return_tensors="pt", truncation=True, max_length=2048).to(explainer_model.device)
        with torch.no_grad():
            out = explainer_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.0,
                num_beams=1,
            )
        text = explainer_tok.decode(out[0], skip_special_tokens=True)
        return text[len(prompt):].strip() if text.startswith(prompt) else text.strip()
    else:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        inputs = explainer_tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
        with torch.no_grad():
            out = explainer_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=4)
        return explainer_tok.decode(out[0], skip_special_tokens=True).strip()


Cell 6 — Build Misconception Knowledge Base (KB) using the LLM (cached)

In [ ]:
import json, re
from collections import Counter

train_df["miscon_only"] = train_df[mis_col].astype(str)
freq = Counter(train_df["miscon_only"].tolist())
top_miscons = [m for m,_ in freq.most_common(KB_BUILD_TOP_N)]
print("KB target:", len(top_miscons), "most common:", freq.most_common(5))

if KB_PATH.exists():
    kb = json.loads(KB_PATH.read_text(encoding="utf-8"))
    print("✅ Loaded KB entries:", len(kb))
else:
    kb = {}
    for i, miscon in enumerate(top_miscons, start=1):
        sub = train_df[train_df["miscon_only"] == miscon]
        sample = sub.sample(n=min(KB_EXAMPLES_PER_MISCON, len(sub)), random_state=42)

        examples = []
        for _, r in sample.iterrows():
            examples.append({
                "question": str(r[qtext_col]) if qtext_col else "",
                "student_answer": str(r[ans_col]) if ans_col else "",
                "student_expl": str(r[exp_col]) if exp_col else "",
            })

        prompt = f"""You are a math teacher writing a misconception knowledge card.

Misconception name: {miscon}

Student examples (question + reasoning):
{json.dumps(examples, ensure_ascii=False, indent=2)}

Return ONLY JSON with keys:
plain_definition (1-2 sentences),
typical_wrong_thought (list),
correct_rule (list),
fix_steps (list of 4-7 steps),
quick_check (one question),
tiny_example (2-4 lines).
"""
        out = llm_generate(prompt, max_new_tokens=420) if explainer_model is not None else ""
        card = None
        if out:
            m = re.search(r"\{.*\}", out, flags=re.S)
            if m:
                try:
                    card = json.loads(m.group(0))
                except Exception:
                    card = None

        if card is None:
            card = {
                "plain_definition": f"Students often make this mistake: {miscon}.",
                "typical_wrong_thought": ["They apply an incorrect rule or shortcut."],
                "correct_rule": ["Use the correct definition and check carefully."],
                "fix_steps": ["Restate the goal.", "Write the correct rule.", "Apply it step-by-step.", "Check with an easy example."],
                "quick_check": "Can you test your rule on a simpler number?",
                "tiny_example": "Example: Compare decimals by place value, not by number of digits."
            }

        kb[miscon] = card
        if i % 10 == 0:
            print(f"Built {i}/{len(top_miscons)} cards...")

    KB_PATH.write_text(json.dumps(kb, ensure_ascii=False, indent=2), encoding="utf-8")
    print("✅ Saved KB:", len(kb), "to", KB_PATH)

k0 = next(iter(kb.keys()))
print("Sample KB:", k0)
print(json.dumps(kb[k0], ensure_ascii=False, indent=2)[:1200])


Cell 7 — Explainable AI Feedback Generator + Verifier

In [ ]:
import numpy as np
import re
import json

def parse_top3(s: str):
    parts = str(s).split()
    return parts[:3] if parts else []

def get_correct_choice(qid):
    if qid_col is None or ans_col is None:
        return ""
    return str(correct_by_q_full.get(qid, ""))

def rule_based_verify(feedback: str, is_correct: int, correct_choice: str) -> str:
    txt = feedback.strip()

    # If the student is correct, don't say "wrong/incorrect"
    if is_correct == 1:
        txt = re.sub(r"\bincorrect\b", "not quite right", txt, flags=re.I)
        txt = re.sub(r"\bwrong\b", "not quite right", txt, flags=re.I)

    # If student is incorrect and correct choice is known, ensure we mention it
    if is_correct == 0 and correct_choice and correct_choice not in txt:
        txt = f"✅ Correct answer choice: {correct_choice}\n\n" + txt

    return txt

def llm_verify_and_fix(feedback: str) -> str:
    if (not USE_LLM_VERIFIER) or (explainer_model is None):
        return feedback

    prompt = f"""You are a strict teacher-editor.

Fix contradictions and make the feedback concise (8–14 lines), student-friendly, and correct.

Feedback:
{feedback}
"""
    out = llm_generate(prompt, max_new_tokens=260)
    return out if out else feedback

def generate_teacher_feedback(row, pred_label: str, pred_top3: list) -> str:
    qtext = str(row[qtext_col]) if qtext_col else ""
    student_answer = str(row[ans_col]) if ans_col else ""
    student_expl = str(row[exp_col]) if exp_col else ""
    qid = row[qid_col] if qid_col else None

    correct_choice = get_correct_choice(qid) if qid is not None else ""
    ic = int(row["is_correct"]) if "is_correct" in row else -1
    ic_str = {1:"Yes", 0:"No"}.get(ic, "Unknown")

    miscon_only = pred_label.split(":", 1)[1] if ":" in pred_label else pred_label
    card = kb.get(miscon_only, {})
    card_json = json.dumps(card, ensure_ascii=False, indent=2)

    if USE_LLM_EXPLAINER and explainer_model is not None:
        prompt = f"""You are a kind, clear math teacher.

Use ONLY the info below. Do not invent details.

Question: {qtext}
Student chose: {student_answer}
Is correct? {ic_str}
Correct choice (if known): {correct_choice}
Student explanation: {student_expl}

Top predictions: {pred_top3}

Misconception knowledge card (JSON):
{card_json}

Write:
- 1–2 lines: what the student seems to think
- 1 line: misconception in simple words
- 4–7 short steps: correct logic
- 1 line: self-check question

Keep it concise and encouraging.
"""
        fb = llm_generate(prompt, max_new_tokens=320) or "LLM failed to generate feedback."
    else:
        fb = f"""Teacher Feedback (template)
You chose: {student_answer}. Correct? {ic_str}. Correct choice: {correct_choice}
Likely misconception: {pred_label}

Steps:
1) Restate what is asked.
2) Write the correct rule.
3) Apply it step-by-step.
4) Check using a simple example.

Self-check: Does your rule still work if numbers change slightly?
"""

    fb = rule_based_verify(fb, ic, correct_choice)
    fb = llm_verify_and_fix(fb)
    return fb.strip()

# Demo: show explanations for 5 random rows
idxs = np.random.default_rng(42).choice(len(train_df), size=5, replace=False)
for i in idxs:
    row = train_df.iloc[i]
    top3 = parse_top3(row["pred_top3"])
    pred1 = top3[0] if top3 else "Unknown:Unknown"
    print("="*100)
    print("Row:", i)
    print("True:", row["label_str"])
    print("Pred:", top3)
    print()
    print(generate_teacher_feedback(row, pred1, top3))
    print()
